In [3]:
import torch
import torch.nn.functional as F
import random
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
words = open('names.txt', 'r').read().splitlines()

In [5]:
# from/to int lookup table
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

In [131]:
# hyper-parameters
block_size = 4
emb_dim = 10
num_neurons = 300
mini_batch_size = 32

In [132]:
# creating data set

def build_dataset(word):
  X, Y = [], []
  for w in word:
    context = [0] * block_size
    # print(w)
    for ch in list(w) + ['.']:
      ix = stoi[ch]
      Y.append(ix)
      X.append(context)
      context = context[1:] + [ix]
      # print((''.join(itos[i] for i in context)),'---->',itos[ix])
  return X, Y


# train, dev, test split
random.seed(43)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

# train data set
Xtr = torch.tensor(build_dataset(words[:n1])[0])
Ytr = torch.tensor(build_dataset(words[:n1])[1])

# dev data set
Xdev = torch.tensor(build_dataset(words[n1:n2])[0])
Ydev = torch.tensor(build_dataset(words[n1:n2])[1])

# test dev set
Xtest = torch.tensor(build_dataset(words[n2:])[0])
Ytest = torch.tensor(build_dataset(words[n2:])[1])

In [133]:
Xtr.shape

torch.Size([182516, 4])

In [134]:
Ytr.shape

torch.Size([182516])

In [135]:
num_examples = Xtr.shape[0]

In [139]:
# initialize network
g = torch.Generator().manual_seed(2142355235)
C = torch.randn((27, emb_dim), generator=g) # (27, 2)
W1 = torch.randn((block_size*emb_dim, num_neurons), generator=g)
b1 = torch.randn(num_neurons, generator=g)
W2 = torch.randn((num_neurons, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [140]:
for p in parameters:
  p.requires_grad=True

In [141]:
# training on Xtr data
num_iterations = 200000

for i in range(num_iterations):

  # creating mini batch
  m_indices = torch.randint(0, Xtr.shape[0], (mini_batch_size, ))

  X_mini_batch = Xtr[m_indices]
  Y_mini_batch = Ytr[m_indices]

  emb = C[X_mini_batch].view(mini_batch_size, block_size * emb_dim)

  # forward pass
  h = torch.tanh(emb @ W1 + b1)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, Y_mini_batch)


  # backward pass
  for p in parameters:
    p.grad = None

  loss.backward()

  # calculating learning rate with decay
  x = i / num_iterations
  lr = 0.1 - 0.099 * torch.exp(torch.tensor(10 * (x - 1)))

  # update
  for p in parameters:
    p.data += -lr * p.grad

print(loss.item())


2.1334850788116455


In [142]:
# total train loss
emb = C[Xtr].view(num_examples, block_size * emb_dim)
h = torch.tanh(emb @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytr)
print(loss.item())

2.0700254440307617


In [156]:
# total dev loss
emb = C[Xdev].view(Xdev.shape[0], block_size * emb_dim)
h = torch.tanh(emb @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ydev)
print(loss.item())

2.1366522312164307


In [162]:
# sampling

for _ in range(10):

  out = []
  context = [0] * block_size

  while True:
    emb = C[context].view(1, block_size * emb_dim)
    h = torch.tanh(emb @ W1 + b1)
    logits = h @ W2 + b2
    prob = F.softmax(logits,dim=1)
    ix = torch.multinomial(prob, num_samples=1, generator=g).item()
    out.append(itos[ix])
    context = context[1:] + [ix]
    if ix == 0:
      break

  print(''.join(c for c in out))




meoruweyna.
ciyco.
layi.
maxi.
korie.
jarrion.
mamrou.
olabacari.
kassen.
natiyann.
